<a href="https://colab.research.google.com/github/amoeba-aoi/ImmuScope_reproduction_and_extension/blob/main/immuscope_reproduction_different_division.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 挂载 Drive，用于备份
from pathlib import Path
import os

print('1 use Drive')
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# 2 若不存在仓库则从github克隆
print('2 clone github')
%cd /content
if not Path("/content/ImmuScope").exists():
    !git clone https://github.com/shenlongchen/ImmuScope.git
%cd /content/ImmuScope

# 3 安装依赖
print('3 install dependency')
!pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "pandas==2.2.2" \
  "scikit-learn==1.4.2" \
  "h5py==3.11.0" \
  "click==8.0.4"\
  tqdm \
  ruamel.yaml

import numpy, pandas, sklearn, h5py, click
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("sklearn", sklearn.__version__)
print("h5py", h5py.__version__)
print("click", click.__version__)

# 4 优先从 Drive 恢复数据与权重，不存在则下载
print('4 recover data and weight')
import shutil

DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/ImmuScope_backup")  # 可改成你的实际备份目录
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    "ImmuScope-data.tar.gz": "https://zenodo.org/records/14810445/files/ImmuScope-data.tar.gz?download=1",
    "ImmuScope-weights.tar.gz": "https://zenodo.org/records/14810445/files/ImmuScope-weights.tar.gz?download=1",
}

def gzip_ok(fp: Path) -> bool:
    if (not fp.exists()) or fp.stat().st_size == 0:
        return False
    # 返回码 0 表示 gzip 完整
    return os.system(f'gzip -t "{fp}" >/dev/null 2>&1') == 0

for name, url in artifacts.items():
    local_fp = Path(name)
    drive_fp = DRIVE_BACKUP_DIR / name

    # A) 本地包完整 -> 直接用
    if gzip_ok(local_fp):
        print(f"[OK] local exists: {local_fp}")
        continue

    # B) 本地包损坏则删除
    if local_fp.exists():
        local_fp.unlink()
        print(f"[CLEAN] removed broken local: {local_fp}")

    # C) Drive 包完整 -> 恢复到本地
    if gzip_ok(drive_fp):
        shutil.copy2(drive_fp, local_fp)
        print(f"[RESTORE] from Drive: {drive_fp}")
    else:

        # D) Drive 不存在可用包 -> 下载
        print(f"[DOWNLOAD] {name}")
        !wget -c "{url}" -O "{name}"

        if not gzip_ok(local_fp):
            raise RuntimeError(f"{name} 下载后仍不完整，请重试。")

# 再次完整性测试
!gzip -t ImmuScope-data.tar.gz && echo "data tar.gz OK"
!gzip -t ImmuScope-weights.tar.gz && echo "weights tar.gz OK"

# 查看包内结构
!tar -tzf ImmuScope-data.tar.gz | head -n 20
!tar -tzf ImmuScope-weights.tar.gz | head -n 20

# 解压
!tar -xzf ImmuScope-data.tar.gz -C .
!tar -xzf ImmuScope-weights.tar.gz -C .

# 快速确认
!ls -lah data/raw | head
!ls -lah data/train_test_h5py | head

1 use Drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2 clone github
/content
/content/ImmuScope
3 install dependency
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 283.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 338.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 364.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 388.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 273.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 268.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.5/97.5 kB 384.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 364.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 422.6 MB/s eta 0:

In [ ]:
# 5 写 configs/data.yaml
config_content = """mhc_seq: data/raw/pseudosequence.2023.dat
dataset_id_mhc: data/raw/allelelist
dataset_ms: data/tmp_datasets
5cv_ma: data/5cv_MA_h5py/5cv_.h5
5cv_sa: data/5cv_SA_h5py/5cv_.h5
train_sa: data/train_test_h5py/sa_train.h5
train_ma: data/train_test_h5py/ma_train.h5
train_ba: data/train_test_h5py/ba_train.h5
test: data/train_test_h5py/NetMHCIIpan_eval.h5
train_imm: data/imm/imm_train.h5
test_imm: data/imm/imm_test.h5
logs: results/logs
results: results
"""
Path("configs").mkdir(parents=True, exist_ok=True)
with open("configs/data.yaml", "w") as f:
    f.write(config_content)

print("configs/data.yaml 已写入")
!ls -lah data/train_test_h5py/

# main_antigen_presentation_train.py 中
# train_path_ms = Path(os.path.join(data_cnf["dataset_ms"], f"{model_name}_{model_id}_train.h5"))
# res_path_with_id = Path(res_path, f'{model_name}-{model_id}')
# 其他目录在代码中有显式创建，但dataset_ms没有使用mkdir创建目录，导致往其中写入文件时报错
# 修复代码，可作为工程改进点
# reproducibility / robustness enhancement（提升可复现性和稳定性）
!mkdir -p data/tmp_datasets
!ls -ld data/tmp_datasets

# 快速验证 configs/data.yaml 路径是否可用
from pathlib import Path
import yaml

cfg_path = Path("configs/data.yaml")
assert cfg_path.exists(), f"未找到配置文件: {cfg_path}"

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(f"读取配置: {cfg_path}\n")

# 1) 先确保日志目录存在
Path(cfg.get("logs", "results/logs")).mkdir(parents=True, exist_ok=True)
Path(cfg.get("dataset_ms", "data/tmp_datasets")).mkdir(parents=True, exist_ok=True)

# 2) 常规路径检查
skip_keys = {"5cv_ma", "5cv_sa"}
ok, bad = [], []
for k, v in cfg.items():
    if k in skip_keys:
        continue
    p = Path(v)
    if p.exists():
        ok.append((k, v))
    else:
        bad.append((k, v))

print("=== 常规路径检查 ===")
for k, v in ok:
    print(f"[OK ] {k:12s} -> {v}")
for k, v in bad:
    print(f"[MISS] {k:12s} -> {v}")

# 3) 5cv 专项检查：检查展开后的真实文件
print("\n=== 5CV 文件检查 ===")
cv_bad = []
for key in ["5cv_ma", "5cv_sa"]:
    templ = cfg.get(key)
    if not templ:
        cv_bad.append((key, "未配置"))
        continue

    # 期望模板形如 .../5cv_.h5
    files_to_check = []
    if key == "5cv_ma":
        files_to_check = [templ.replace("_.h5", f"_{i}_train.h5") for i in range(5)]
    else:  # 5cv_sa
        files_to_check = (
            [templ.replace("_.h5", f"_{i}_train.h5") for i in range(5)] +
            [templ.replace("_.h5", f"_{i}_test.h5") for i in range(5)]
        )

    missing = [p for p in files_to_check if not Path(p).exists()]
    if missing:
        cv_bad.append((key, missing[:3]))  # 只展示前3个
        print(f"[MISS] {key} 缺失 {len(missing)} 个文件，例如: {missing[:3]}")
    else:
        print(f"[OK ] {key} 5-fold 文件齐全")

# 4) 关键文件检查
critical = [
    "data/raw/pseudosequence.2023.dat",
    "data/raw/allelelist",
    "data/train_test_h5py/sa_train.h5",
    "data/train_test_h5py/ma_train.h5",
    "data/train_test_h5py/ba_train.h5",
    "data/train_test_h5py/NetMHCIIpan_eval.h5",
    "data/imm/imm_train.h5",
    "data/imm/imm_test.h5",
]
print("\n=== 关键文件检查 ===")
critical_bad = []
for p in critical:
    exists = Path(p).exists()
    print(f"{'OK  ' if exists else 'MISS'} {p}")
    if not exists:
        critical_bad.append(p)

# 5) 结果输出
if bad or critical_bad:
    raise FileNotFoundError(
        f"常规缺失 {len(bad)} 项, 关键缺失 {len(critical_bad)} 项，请修正后再训练。"
    )
else:
    print("\n常规训练路径验证通过。")

if cv_bad:
    print("\n提示：5cv 相关文件不完整，当前不适合跑 main_antigen_presentation_5cv.py")
else:
    print("5cv 路径验证通过，可跑 5cv。")

configs/data.yaml 已写入
total 4.0G
drwxrwxr-x 3 1000 1000 4.0K Nov 18  2024 .
drwxrwxr-x 8 1000 1000 4.0K Nov 18  2024 ..
-rw-rw-r-- 1 1000 1000  26M Nov 17  2024 ba_train.h5
-rw-rw-r-- 1 1000 1000 3.3G Nov 17  2024 ma_train.h5
drwxrwxr-x 2 1000 1000 4.0K Nov 19  2024 ms_tmp_dataset
-rw-rw-r-- 1 1000 1000 164M Nov 17  2024 NetMHCIIpan_eval.h5
-rw-rw-r-- 1 1000 1000 545M Nov 17  2024 sa_train.h5
drwxr-xr-x 2 root root 4096 Apr 11 13:28 data/tmp_datasets
读取配置: configs/data.yaml

=== 常规路径检查 ===
[OK ] mhc_seq      -> data/raw/pseudosequence.2023.dat
[OK ] dataset_id_mhc -> data/raw/allelelist
[OK ] dataset_ms   -> data/tmp_datasets
[OK ] train_sa     -> data/train_test_h5py/sa_train.h5
[OK ] train_ma     -> data/train_test_h5py/ma_train.h5
[OK ] train_ba     -> data/train_test_h5py/ba_train.h5
[OK ] test         -> data/train_test_h5py/NetMHCIIpan_eval.h5
[OK ] train_imm    -> data/imm/imm_train.h5
[OK ] test_imm     -> data/imm/imm_test.h5
[OK ] logs         -> results/logs
[OK ] results   

In [ ]:
# 确认路径数据存在：
import os, yaml
with open("configs/data.yaml") as f:
    d = yaml.safe_load(f)

keys = ["train_sa", "train_ma", "mhc_seq", "dataset_ms"]
for k in keys:
    p = d[k]
    print(k, p, "=>", os.path.exists(p))

train_sa data/train_test_h5py/sa_train.h5 => True
train_ma data/train_test_h5py/ma_train.h5 => True
mhc_seq data/raw/pseudosequence.2023.dat => True
dataset_ms data/tmp_datasets => True


In [ ]:
# 从drive恢复tiny文件
# 根据需要修改
!tar -xzf '/content/drive/MyDrive/ImmuScope_tiny_backup/ImmuScope_tiny_h5_20260402_140114.tar.gz' -C /content/ImmuScope

In [ ]:
# 另存最小配置文件，仅用于跑通
%cd /content/ImmuScope
import yaml

with open("configs/ImmuScope-EL.yaml") as f:
    c = yaml.safe_load(f)

c["name"] = "ImmuScope-EL-min"
c["train"]["pretrain_epochs"] = 1
c["train"]["sample_incorporate_epochs"] = 0
c["train"]["fine_tune_epochs"] = 0
c["train"]["batch_size"] = 256
c["train"]["num_workers"] = 0
c["valid"]["batch_size"] = 256
c["test"]["batch_size"] = 256

with open("configs/ImmuScope-EL-min.yaml", "w") as f:
    yaml.safe_dump(c, f, sort_keys=False)

print("CHECK train cfg:", c["train"])

/content/ImmuScope
CHECK train cfg: {'pretrain_epochs': 1, 'sample_incorporate_epochs': 0, 'fine_tune_epochs': 0, 'batch_size': 256, 'opt_params': {'lr_pre': 0.001, 'wd_pre': '1e-4', 'lr_si': '3e-5', 'wd_si': '1e-4', 'lr_finetune': '5e-5', 'wd_finetune': '1e-4'}, 'num_workers': 0}


In [ ]:
# Dataset 类在初始化时会把整份 H5 一次性读入内存（尤其 MA 很大），所以进程很容易被系统 Killed
# 生成 tiny train SA/MA
%cd /content/ImmuScope
import h5py, numpy as np, os

os.makedirs("data/tiny", exist_ok=True)

def make_tiny(src, dst, n_rows, bag_size=None):
    with h5py.File(src, "r") as f, h5py.File(dst, "w") as g:
        n = min(n_rows, len(f["labels"]))
        if bag_size is not None:
            n = (n // bag_size) * bag_size  # MA 要对齐 bag_size
        for k in f.keys():
            g.create_dataset(k, data=f[k][:n], dtype=f[k].dtype)
        print(dst, "rows:", n)

make_tiny("data/train_test_h5py/sa_train.h5", "data/tiny/sa_train_tiny.h5", n_rows=200000)
make_tiny("data/train_test_h5py/ma_train.h5", "data/tiny/ma_train_tiny.h5", n_rows=200000, bag_size=10)

# 临时改 data.yaml 的 EL 路径，最小侵入式排障：改配置，不改模型算法，绕开 Colab 内存限制（EXIT_CODE:137）
# 目标是先让 EL 流程跑通，而不是马上跑完整大数据
# smoke test（功能验证）：证明流程可运行，不是最终 benchmark 结果

import yaml
with open("configs/data.yaml") as f:
    d = yaml.safe_load(f)

d["train_sa"] = "data/tiny/sa_train_tiny.h5"
d["train_ma"] = "data/tiny/ma_train_tiny.h5"

with open("configs/data-el-tiny.yaml", "w") as f:
    yaml.safe_dump(d, f, sort_keys=False)

print("wrote configs/data-el-tiny.yaml")

/content/ImmuScope
data/tiny/sa_train_tiny.h5 rows: 200000
data/tiny/ma_train_tiny.h5 rows: 200000
wrote configs/data-el-tiny.yaml


In [ ]:
# EL训练异常退出定位单元
%cd /content/ImmuScope
import yaml, torch
from torch.utils.data import DataLoader
from ImmuScope.datasets.datasets import MABags, SinInstanceBag
from ImmuScope.utils.data_utils import create_splits_train_valid_test, get_mhc_name_seq

with open("configs/data.yaml") as f:
    data_cnf = yaml.safe_load(f)
with open("configs/ImmuScope-EL-min.yaml") as f:
    model_cnf = yaml.safe_load(f)

print("STEP 1: load mhc seq")
mhc_name_seq = get_mhc_name_seq(data_cnf["mhc_seq"])
print("  mhc loaded:", len(mhc_name_seq))

print("STEP 2: split SA")
train_idx, valid_idx, test_idx = create_splits_train_valid_test(
    data_cnf["train_sa"], train_ratio=0.1, valid_ratio=0.05, test_ratio=0.05, seed=model_cnf["seed"]
)
print("  split lens:", len(train_idx), len(valid_idx), len(test_idx))

print("STEP 3: build datasets")
ds_sa = SinInstanceBag(data_cnf["train_sa"], mhc_name_seq, indices=train_idx)
ds_ma = MABags(data_cnf["train_ma"], mhc_name_seq, model_cnf["model"]["bag_size"])
print("  ds_sa:", len(ds_sa), "ds_ma:", len(ds_ma))

print("STEP 4: build dataloaders")
dl_sa = DataLoader(ds_sa, batch_size=model_cnf["train"]["batch_size"] * 10, shuffle=True, drop_last=True, num_workers=0)
dl_ma = DataLoader(ds_ma, batch_size=model_cnf["train"]["batch_size"], shuffle=True, drop_last=True, num_workers=0)
print("  dl_sa len:", len(dl_sa), "dl_ma len:", len(dl_ma))

print("STEP 5: fetch one batch")
b_sa = next(iter(dl_sa))
b_ma = next(iter(dl_ma))
print("  first batch ok")
print("DONE")

In [ ]:
# 修改main_antigen_presentation_train.py 中train集过大问题
# create_splits_train_valid_test函数实现：train 用 train_ratio，valid 用 valid_ratio，test 用剩下全部
# 修复代码，可作为工程改进点
# reproducibility / robustness enhancement
from pathlib import Path

p = Path("/content/ImmuScope/main_antigen_presentation_train.py")
s = p.read_text()
s = s.replace("train_ratio=0.1, valid_ratio=0.05,",
              "train_ratio=0.9, valid_ratio=0.05,")
p.write_text(s)
print("patched:", p)

patched: /content/ImmuScope/main_antigen_presentation_train.py


In [ ]:
# 修改main_antigen_presentation_train.py 中最小运行模式中修改了参数fine_tune_epochs = 0 导致要找fine-tune-b.pt的情况
# 强制在脚本末尾的测试 / output_res 总是加载 -pretrain.pt，而不会加载 -fine-tune-b.pt
# fine_tune_b >0 时不适用，仅适用smoke test

%cd /content/ImmuScope
from pathlib import Path
import re

p = Path("main_antigen_presentation_train.py")
s = p.read_text()

# 强制重写两个测试函数
s = re.sub(
    r"def test_immuscope_el\(trainer, model_cnf, test_path, mhc_name_seq\):[\s\S]*?def test_immuscope_el_with_loader",
    """def test_immuscope_el(trainer, model_cnf, test_path, mhc_name_seq):
    test_loader = DataLoader(SinInstanceBag(test_path, mhc_name_seq, indices=None),
                             batch_size=model_cnf['test']['batch_size'])
    pred_instances, pred_bags, _ = trainer.predict(test_loader, model_prefix='pretrain')
    return pred_instances, pred_bags, test_loader.dataset.labels, test_loader.dataset.mhc_names

def test_immuscope_el_with_loader""",
    s
)

s = re.sub(
    r"def test_immuscope_el_with_loader\(trainer, test_loader\):[\s\S]*?@click\.command\(\)",
    """def test_immuscope_el_with_loader(trainer, test_loader):
    pred_instances, pred_bags, _ = trainer.predict(test_loader, model_prefix='pretrain')
    return (pred_instances, pred_bags, test_loader.dataset.labels[test_loader.dataset.indices],
            test_loader.dataset.mhc_names[test_loader.dataset.indices])


@click.command()""",
    s
)

p.write_text(s)
print("patched")

# 语法检查
!python -m py_compile /content/ImmuScope/main_antigen_presentation_train.py && echo "syntax ok"

/content/ImmuScope
patched
syntax ok


In [ ]:
# 完成修改tiny SA、MA数据集，修改daya.yaml的EL路径后跑EL smoke test

# 防止混淆：
print("DATA_CNF = configs/data-el-tiny.yaml")
print("MODEL_CNF = configs/ImmuScope-EL-min.yaml")

# 跑EL
%cd /content/ImmuScope
!python -u main_antigen_presentation_train.py \
  --data-cnf configs/data-el-tiny.yaml \
  --model-cnf configs/ImmuScope-EL-min.yaml \
  --start-id 0 --num_models 1

In [ ]:
# EL训练异常中断打印 ^C code
%cd /content/ImmuScope
!bash -lc 'python -u main_antigen_presentation_train.py --data-cnf configs/data.yaml --model-cnf configs/ImmuScope-EL-min.yaml --start-id 0 --num_models 1; echo EXIT_CODE:$?'

In [ ]:
# 同上，生成 tiny 5cv SA/MA + 写 data-el-tiny.yaml
%cd /content/ImmuScope
import os, h5py, yaml
from pathlib import Path

# ====== 真实源路径 ======
SRC_SA = "data/5cv_SA_h5py"
SRC_MA = "data/5cv_MA_h5py"

# ====== 输出路径 ======
DST = "data/tiny_5cv"
BAG = 10
N_SA = 30000      # 防 OOM
N_MA = 100000     # BAG 的整数倍

os.makedirs(f"{DST}/5cv_sa_h5py", exist_ok=True)
os.makedirs(f"{DST}/5cv_ma_h5py", exist_ok=True)

'''
def slice_h5(src, dst, n, bag_align=None):
    with h5py.File(src, "r") as f, h5py.File(dst, "w") as g:
        n = min(n, len(f["labels"]))
        if bag_align:
            n = (n // bag_align) * bag_align
        for k in f.keys():
            g.create_dataset(k, data=f[k][:n])
    print("OK:", dst, "rows:", n)
'''

# triplet loss 的批次标签单一（全 1 或全 0）
# 将slice_h5 改成随机采样版
# 工程改进点
import numpy as np

def slice_h5_random(src, dst, n, bag_align=None, seed=2026):
    rng = np.random.default_rng(seed)
    with h5py.File(src, "r") as f, h5py.File(dst, "w") as g:
        total = len(f["labels"])
        n = min(n, total)
        if bag_align:
            # 对 MA：按 bag 抽样，避免打乱 bag 结构
            # 因为 MA 是按 bag_size=10 组织，直接随机行会打乱 bag 结构
            bag_size = bag_align
            n_bags_total = total // bag_size
            n_bags = min(n // bag_size, n_bags_total)
            bag_ids = rng.choice(n_bags_total, size=n_bags, replace=False)
            bag_ids.sort()
            idx = np.concatenate([np.arange(b*bag_size, (b+1)*bag_size) for b in bag_ids])
        else:
            idx = rng.choice(total, size=n, replace=False)
            idx.sort()

        for k in f.keys():
            g.create_dataset(k, data=f[k][idx])

        print(dst, "rows:", len(idx))

for cv in range(5):
    # SA: train + test
    slice_h5_random(f"{SRC_SA}/5cv_{cv}_train.h5", f"{DST}/5cv_sa_h5py/5cv_{cv}_train.h5", N_SA)
    slice_h5_random(f"{SRC_SA}/5cv_{cv}_test.h5",  f"{DST}/5cv_sa_h5py/5cv_{cv}_test.h5",  N_SA)

    # MA: only train
    slice_h5_random(f"{SRC_MA}/5cv_{cv}_train.h5", f"{DST}/5cv_ma_h5py/5cv_{cv}_train.h5", N_MA, BAG)

# 写 tiny 专用 data config
with open("configs/data.yaml", "r") as f:
    d = yaml.safe_load(f)

d["5cv_sa"] = "data/tiny_5cv/5cv_sa_h5py/5cv_.h5"
d["5cv_ma"] = "data/tiny_5cv/5cv_ma_h5py/5cv_.h5"

# 如果已有 train tiny，可一起用
if Path("data/tiny/sa_train_tiny.h5").exists():
    d["train_sa"] = "data/tiny/sa_train_tiny.h5"
if Path("data/tiny/ma_train_tiny.h5").exists():
    d["train_ma"] = "data/tiny/ma_train_tiny.h5"

out = Path("configs/data-el-tiny.yaml")
with open(out, "w") as f:
    yaml.safe_dump(d, f, sort_keys=False, allow_unicode=True)

print("\nWrote:", out)
print("5cv_sa:", d["5cv_sa"])
print("5cv_ma:", d["5cv_ma"])

In [ ]:
# 修改main_antigen_presentation_5cv.py在最小运行模式中修改了参数sample_incorporate_epochs=0、fine_tune_epochs=0导致要找fine-tune-b.pt的情况
# 强制将所有test都读成pretrain
# fine_tune_b >0 时不适用，仅适用smoke test
%cd /content/ImmuScope
from pathlib import Path
p = Path("main_antigen_presentation_5cv.py")
s = p.read_text()
s = s.replace(
  "pred_instances, pred_bags, _ = trainer.predict(test_loader)",
  "pred_instances, pred_bags, _ = trainer.predict(test_loader, model_prefix='pretrain')"
)
p.write_text(s)
print("patched", p)

/content/ImmuScope
patched main_antigen_presentation_5cv.py


In [ ]:
# 恢复原main_antigen_presentation_5cv.py
%cd /content/ImmuScope
!git checkout -- main_antigen_presentation_5cv.py

/content/ImmuScope


In [ ]:
# 恢复原main_antigen_presentation_train.py
# 去除划分比例修改和正则补丁，划分比例main和data_utils的不一致在后续论文中说明
%cd /content/ImmuScope
!git checkout -- main_antigen_presentation_train.py

# 重新做data/tmp_datasets的显式创建
!mkdir -p data/tmp_datasets
!ls -ld data/tmp_datasets

/content/ImmuScope
drwxr-xr-x 2 root root 4096 Apr  5 03:34 data/tmp_datasets


In [ ]:
# 将修改后的main_antigen_presentation_train.py和main_antigen_presentation_5cv.py备份到drive
%cd /content/ImmuScope
from google.colab import drive
from pathlib import Path
import shutil
from datetime import datetime

drive.mount("/content/drive")

PROJECT = Path("/content/ImmuScope")
DST_ROOT = Path("/content/drive/MyDrive/ImmuScope_code_backup")
DST_ROOT.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DST = DST_ROOT / f"main_scripts_{stamp}"
DST.mkdir(parents=True, exist_ok=True)

files = [
    PROJECT / "main_antigen_presentation_train.py",
    PROJECT / "main_antigen_presentation_5cv.py",
]

for f in files:
    if not f.exists():
        raise FileNotFoundError(f"找不到: {f}")
    shutil.copy2(f, DST / f.name)
    print("已备份:", f.name, "->", DST / f.name)

print("\n备份目录:", DST)

/content/ImmuScope
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
已备份: main_antigen_presentation_train.py -> /content/drive/MyDrive/ImmuScope_code_backup/main_scripts_20260405_042522/main_antigen_presentation_train.py
已备份: main_antigen_presentation_5cv.py -> /content/drive/MyDrive/ImmuScope_code_backup/main_scripts_20260405_042522/main_antigen_presentation_5cv.py

备份目录: /content/drive/MyDrive/ImmuScope_code_backup/main_scripts_20260405_042522


In [ ]:
# 完成修改tiny SA、MA数据集，修改daya.yaml的EL路径后跑5cv的smoke test

# 防止混淆：
print("DATA_CNF = configs/data-el-tiny.yaml")
print("MODEL_CNF = configs/ImmuScope-EL-min.yaml")

%cd /content/ImmuScope
# 0) 语法检查（train + 5cv）
!python -m py_compile main_antigen_presentation_train.py main_antigen_presentation_5cv.py && echo "syntax ok"
# 1) 快速检查 tiny 5cv 文件是否都存在
import os
ok = True
for cv in range(5):
    for p in [
        f"data/tiny_5cv/5cv_sa_h5py/5cv_{cv}_train.h5",
        f"data/tiny_5cv/5cv_sa_h5py/5cv_{cv}_test.h5",
        f"data/tiny_5cv/5cv_ma_h5py/5cv_{cv}_train.h5",
    ]:
        if not os.path.exists(p):
            ok = False
            print("MISSING:", p)

print("tiny 5cv files ready:", ok)
if not ok:
    raise RuntimeError("tiny 5cv files are missing, stop before training.")
# 2) 跑 5cv
!python -u main_antigen_presentation_5cv.py \
  --data-cnf configs/data-el-tiny.yaml \
  --model-cnf configs/ImmuScope-EL-min.yaml \
  --start-id 0 --num_models 1

DATA_CNF = configs/data-el-tiny.yaml
MODEL_CNF = configs/ImmuScope-EL-min.yaml
/content/ImmuScope
syntax ok
tiny 5cv files ready: True
[I 260402 14:03 utils:64] Model Name: ImmuScope-EL-min
[I 260402 14:03 main_antigen_presentation_5cv:78] ------------- Start training model_id: 0 - cv: 0 ------------
Training: 100% 39/39 [00:14<00:00,  2.71it/s]
[D 260402 14:03 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0065 AUPR: 0.1138 PPV: 0.1301
[D 260402 14:04 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0055 AUPR: 0.1180 PPV: 0.1093
[I 260402 14:04 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.3112 SA_B 0.3367 MA 0.5874 Tri 0.0615] -VAL 0.6313 - Valid:[AUC0_1:0.0104 AUPR:0.1345 PPV:0.1710] -- Test: AUPR: 0.1171 -Group [AUC0_1: 0.0061, AUPR: 0.1307, PPV: 0.1362]
[I 260402 14:04 trainer_el:137] Best Pretrain Epoch: 0

[I 260402 14:04 trainer_el:338] ==== Model loaded from weights/EL/ImmuScope-EL-min-0-CV0-pretrain.pt ====
[D 260402 14:04 trainer_el:190]  ============== Valid Bag:

In [ ]:
# 备份所有 tiny 的 h5（及目录结构）到 Google Drive
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
from datetime import datetime

drive.mount("/content/drive")

PROJECT = Path("/content/ImmuScope")
# Drive 里保存位置（可改）
DRIVE_BACKUP = Path("/content/drive/MyDrive/ImmuScope_tiny_backup")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tar_name = f"ImmuScope_tiny_h5_{stamp}.tar.gz"
tar_path = DRIVE_BACKUP / tar_name

# 要打包的相对路径（在 PROJECT 下）；没有的目录会自动跳过
CANDIDATES = [
    "data/tiny_5cv",
    "data/tiny",
]

to_pack = []
for rel in CANDIDATES:
    p = PROJECT / rel
    if p.exists():
        to_pack.append(rel)
        print("将打包:", p)
    else:
        print("跳过（不存在）:", p)

if not to_pack:
    raise RuntimeError("没有找到任何 tiny 目录（data/tiny_5cv 或 data/tiny）。请先生成 tiny 数据。")

# 在项目根目录执行 tar，保证压缩包内路径为 data/tiny_5cv/...
cmd = ["tar", "-czf", str(tar_path), "-C", str(PROJECT)] + to_pack
subprocess.run(cmd, check=True)
print("\n已生成压缩包:", tar_path)

# 可选：把 yaml 一并拷到同目录（不打进 tar）
for y in ["configs/data-el-tiny.yaml", "configs/ImmuScope-EL-min.yaml"]:
    src = PROJECT / y
    if src.exists():
        shutil.copy2(src, DRIVE_BACKUP / f"{stamp}_{src.name}")
        print("已复制配置:", src.name)

print("\n完成。恢复语句：")
print(f"  !tar -xzf '{tar_path}' -C /content/ImmuScope")

In [ ]:
# 原始配置跑5cv（时间有限只训练一个）
%cd /content/ImmuScope
!python main_antigen_presentation_5cv.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope-EL.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260404 17:09 utils:64] Model Name: ImmuScope-EL
[I 260404 17:09 main_antigen_presentation_5cv:78] ------------- Start training model_id: 0 - cv: 0 ------------
Training: 100% 10155/10155 [30:11<00:00,  5.60it/s]
[D 260404 17:41 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0839 AUPR: 0.8628 PPV: 0.7966
[D 260404 17:41 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0730 AUPR: 0.8042 PPV: 0.7514
[I 260404 17:41 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.1327 SA_B 0.1366 MA 0.2527 Tri 0.0697] -VAL 0.2080 - Valid:[AUC0_1:0.0841 AUPR:0.8635 PPV:0.7971] -- Test: AUPR: 0.8341 -Group [AUC0_1: 0.0748, AUPR: 0.8069, PPV: 0.7525]
Training: 100% 10155/10155 [29:59<00:00,  5.64it/s]
[D 260404 18:11 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0856 AUPR: 0.8775 PPV: 0.8158
[D 260404 18:11 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0753 AUPR: 0.8167 PPV: 0.7626
[I 260404 18:11 trainer_el:451] Epoch-pretrain: 1 - Loss[SA_I 0.1051 SA_B 0.1068 MA 0

In [ ]:
# 恢复无补丁版5cv main后，cv:0 用已有 pretrain，cv:1–4 正常跑
%cd /content/ImmuScope
!python -u main_antigen_presentation_5cv_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-EL.yaml \
  --start-id 0 --num_models 1 \
  --cv-start 0 --cv-end 5 \
  --skip-pretrain-cvs 0

/content/ImmuScope
[I 260405 07:59 utils:64] Model Name: ImmuScope-EL
[I 260405 07:59 main_antigen_presentation_5cv_dongyizhe_20260405:113] ------------- Start training model_id: 0 - cv: 0 ------------
[I 260405 08:01 main_antigen_presentation_5cv_dongyizhe_20260405:126] cv 0: skip_pretrain enabled (load existing *-CV0-pretrain.pt)
[I 260405 08:01 trainer_el:377] ==== Model loaded from weights/EL/ImmuScope-EL-0-CV0-pretrain.pt ====
[I 260405 08:01 trainer_el:170] Skipped MIL pretrain loop (20 epochs); loaded weights/EL/ImmuScope-EL-0-CV0-pretrain.pt

[I 260405 08:01 trainer_el:377] ==== Model loaded from weights/EL/ImmuScope-EL-0-CV0-pretrain.pt ====
[D 260405 08:01 trainer_el:229]  ============== Valid Bag: AUC0_1: 0.0869 AUPR: 0.8879 PPV: 0.8219
[D 260405 08:01 trainer_el:250]  ============== Test Bag: AUC0_1: 0.0750 AUPR: 0.8263 PPV: 0.7681
[I 260405 08:01 trainer_el:500] -- Loss:0.183803 -Valid:[AUC0_1: 0.0869 AUPR: 0.8888 PPV: 0.8233] -- Test: AUPR: 0.8503 -Group [AUC0_1: 0.0769, 

In [ ]:
# 重新划分train valid test比例后跑EL（时间有限只训练一个）
%cd /content/ImmuScope
!python main_antigen_presentation_train.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope-EL.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260411 13:28 utils:64] Model Name: ImmuScope-EL
[I 260411 13:28 main_antigen_presentation_train:74] ------------- Start training model_id: 0 -  ------------
Training: 100% 12468/12468 [34:16<00:00,  6.06it/s]
[D 260411 14:03 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0848 AUPR: 0.8715 PPV: 0.8057
[D 260411 14:03 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0718 AUPR: 0.8488 PPV: 0.7898
[I 260411 14:03 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.1289 SA_B 0.1319 MA 0.2345 Tri 0.0694] -VAL 0.1998 - Valid:[AUC0_1:0.0849 AUPR:0.8730 PPV:0.8049] -- Test: AUPR: 0.8684 -Group [AUC0_1: 0.0701, AUPR: 0.8493, PPV: 0.7898]
Training: 100% 12468/12468 [34:27<00:00,  6.03it/s]
[D 260411 14:38 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0866 AUPR: 0.8869 PPV: 0.8213
[D 260411 14:38 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0711 AUPR: 0.8665 PPV: 0.8087
[I 260411 14:38 trainer_el:451] Epoch-pretrain: 1 - Loss[SA_I 0.1034 SA_B 0.1053 MA 0.16

In [ ]:
# CD4 epitope train
%cd /content/ImmuScope
!python main_cd4_epitope_train.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260412 09:47 utils:64] Model Name: ImmuScope
[I 260412 09:47 main_cd4_epitope_train:21] Start training model weights/CD4/ImmuScope-0.pt
[I 260412 09:48 trainer_cd4_epitope:88] ==== Model loaded from weights/EL/ImmuScope-EL-0-fine-tune-b.pt ====
[I 260412 09:48 trainer_cd4_epitope:164] Epoch: 0 - Loss: BA 0.07343, SA 0.00000 - Valid :[AUC0_1:0.0449 AUPR:0.7478 PPV:0.6817]
[I 260412 09:49 trainer_cd4_epitope:164] Epoch: 1 - Loss: BA 0.03968, SA 0.00000 - Valid :[AUC0_1:0.0477 AUPR:0.7679 PPV:0.6921]
[I 260412 09:49 trainer_cd4_epitope:164] Epoch: 2 - Loss: BA 0.03694, SA 0.00000 - Valid :[AUC0_1:0.0482 AUPR:0.7702 PPV:0.6959]
[I 260412 09:49 trainer_cd4_epitope:164] Epoch: 3 - Loss: BA 0.03535, SA 0.00000 - Valid :[AUC0_1:0.0490 AUPR:0.7736 PPV:0.7011]
[I 260412 09:50 trainer_cd4_epitope:164] Epoch: 4 - Loss: BA 0.03414, SA 0.00000 - Valid :[AUC0_1:0.0496 AUPR:0.7759 PPV:0.7002]
[I 260412 09:50 trainer_cd4_epitope:164] Epoch: 5 - Loss: BA 0.03338, SA 0.00000 - Valid

In [ ]:
# CD4 epitope test
%cd /content/ImmuScope
!python main_cd4_epitope_test.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260412 09:57 utils:64] Model Name: ImmuScope
[I 260412 09:57 trainer_cd4_epitope:88] ==== Model loaded from weights/CD4/ImmuScope-0.pt ====
[I 260412 09:58 main_cd4_epitope_test:49] |**---------Model 0--- TEST: Median AUC: 0.9115; Mean AUC: 0.8160; AVG AUC: 0.7674---------**|
[I 260412 09:58 main_cd4_epitope_test:55] -----------------Average-----------------
[I 260412 09:58 main_cd4_epitope_test:67] |**========== TEST: Median AUC: 0.9115; Mean AUC: 0.8160; AVG AUC: 0.7674==========**|


In [ ]:
# immunogenicity train
%cd /content/ImmuScope
!python main_immunogenicity_train.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope-IM.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260412 09:58 utils:64] Model Name: ImmuScope-IM
[I 260412 09:58 main_immunogenicity_train:20] Start training model weights/IM/ImmuScope-IM-0.pt
[I 260412 09:58 trainer_immunogenicity:84] ==== Model loaded from weights/EL/ImmuScope-EL-0-fine-tune-b.pt ====
Training: 100% 504/504 [00:10<00:00, 49.14it/s]
[I 260412 09:58 trainer_immunogenicity:113] Epoch: 0 - Loss: IMM 0.23357  - Valid-AUC:0.8447  -- Test: [AUC-Group: 0.8416 - AUC-All: 0.8489]
Training: 100% 504/504 [00:09<00:00, 51.33it/s]
[I 260412 09:58 trainer_immunogenicity:113] Epoch: 1 - Loss: IMM 0.20594  - Valid-AUC:0.8590  -- Test: [AUC-Group: 0.8459 - AUC-All: 0.8599]
Training: 100% 504/504 [00:09<00:00, 52.00it/s]
[I 260412 09:59 trainer_immunogenicity:113] Epoch: 2 - Loss: IMM 0.19316  - Valid-AUC:0.8644  -- Test: [AUC-Group: 0.8648 - AUC-All: 0.8704]
Training: 100% 504/504 [00:09<00:00, 52.48it/s]
[I 260412 09:59 trainer_immunogenicity:113] Epoch: 3 - Loss: IMM 0.18046  - Valid-AUC:0.8669  -- Test: [AUC

In [ ]:
# immunogenicity test
%cd /content/ImmuScope
!python main_immunogenicity_test.py \
    --data-cnf configs/data.yaml \
    --model-cnf configs/ImmuScope-IM.yaml \
    --start-id 0 --num_models 1

/content/ImmuScope
[I 260412 10:01 utils:64] Model Name: ImmuScope-IM
[I 260412 10:02 trainer_immunogenicity:84] ==== Model loaded from weights/IM/ImmuScope-IM-0.pt ====
[I 260412 10:02 main_immunogenicity_test:70] |**TEST: AUC_GROUP: 0.8789**|
[I 260412 10:02 main_immunogenicity_test:71] |**TEST: AUC_ALL: 0.8766**|
[I 260412 10:02 main_immunogenicity_test:75] -----------------Average-----------------
[I 260412 10:02 main_immunogenicity_test:85] |**========== TEST: AUC_GROUP: 0.8789 =========**|
[I 260412 10:02 main_immunogenicity_test:86] |**========== TEST: AUC_ALL: 0.8766 =========**|


In [ ]:
# 定时备份
from google.colab import drive
import time

INTERVAL_SECONDS = 600 # 按需修改

drive.mount("/content/drive")

print(f"开始每 {INTERVAL_SECONDS} 秒备份一次，中断请点单元格的停止按钮。")
while True:
    !mkdir -p /content/drive/MyDrive/ImmuScope_change_division_backup
    !cp -r /content/ImmuScope/weights /content/drive/MyDrive/ImmuScope_change_division_backup/
    !cp -r /content/ImmuScope/results /content/drive/MyDrive/ImmuScope_change_division_backup/
    print(f"{time.strftime('%H:%M:%S')} 备份完成，{INTERVAL_SECONDS}s 后下次备份...")
    time.sleep(INTERVAL_SECONDS)